# 04 — VLM Perception Comparison

Compares Vision Language Models for understanding Barcelona Eixample street environments:
- **PerceptionLM** (Meta, local)
- **LLaVA 1.6** (Ollama, local)
- **GPT-4o-Vision** (OpenAI API)
- **MiniCPM-V** (Ollama, local)

**Task**: Given Mapillary street view images, each VLM answers:
1. What type of street environment? (commercial / residential / mixed / park)
2. What amenities are visible?
3. Rate appeal for each archetype (tourist / resident / commuter) 1-5

**Metrics**: Classification accuracy, amenity recall F1, latency, archetype alignment

In [ ]:
import asyncio
import base64
import json
import os
import time
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from openai import AsyncOpenAI

sns.set_theme(style='whitegrid', palette='muted')

# Ground truth labels for test images (add your Mapillary images to data/images/)
GROUND_TRUTH = [
    {'image': 'data/images/eixample_commercial_01.jpg', 'env_type': 'commercial',
     'amenities': ['cafe', 'shop', 'restaurant'], 'tourist_score': 4, 'resident_score': 3, 'commuter_score': 3},
    {'image': 'data/images/eixample_residential_01.jpg', 'env_type': 'residential',
     'amenities': ['pharmacy', 'supermarket'], 'tourist_score': 2, 'resident_score': 5, 'commuter_score': 4},
    {'image': 'data/images/eixample_mixed_01.jpg', 'env_type': 'mixed',
     'amenities': ['cafe', 'pharmacy'], 'tourist_score': 3, 'resident_score': 4, 'commuter_score': 4},
    {'image': 'data/images/eixample_park_01.jpg', 'env_type': 'park',
     'amenities': ['park'], 'tourist_score': 5, 'resident_score': 4, 'commuter_score': 2},
    {'image': 'data/images/eixample_commercial_02.jpg', 'env_type': 'commercial',
     'amenities': ['restaurant', 'bar', 'shop'], 'tourist_score': 5, 'resident_score': 3, 'commuter_score': 2},
]

VLM_PROVIDERS = {
    'llava-ollama': {'base_url': 'http://localhost:11434/v1', 'api_key': 'ollama', 'model': 'llava:13b'},
    'minicpm-ollama': {'base_url': 'http://localhost:11434/v1', 'api_key': 'ollama', 'model': 'minicpm-v'},
    'gpt4o-vision': {'base_url': None, 'api_key': os.getenv('OPENAI_API_KEY', ''), 'model': 'gpt-4o'},
}
# PerceptionLM: add when local endpoint is running
# VLM_PROVIDERS['perceptionlm'] = {'base_url': 'http://localhost:8080/v1', 'api_key': 'local', 'model': 'perception-lm'}

print('VLM providers configured. Add images to data/images/ before running.')

In [ ]:
VISION_PROMPT = """Analyse this street view image from Barcelona Eixample.
Reply with JSON only:
{
  \"env_type\": \"commercial|residential|mixed|park\",
  \"amenities_visible\": [\"list\", \"of\", \"amenity\", \"types\"],
  \"tourist_appeal\": 1-5,
  \"resident_appeal\": 1-5,
  \"commuter_appeal\": 1-5,
  \"reasoning\": \"brief explanation\"
}"""

def encode_image(path):
    with open(path, 'rb') as f:
        return base64.b64encode(f.read()).decode('utf-8')

async def query_vlm(provider_name, config, image_path):
    if not os.path.exists(image_path):
        return None, 0, False
    img_b64 = encode_image(image_path)
    kwargs = {'api_key': config['api_key']}
    if config['base_url']:
        kwargs['base_url'] = config['base_url']
    client = AsyncOpenAI(**kwargs)
    t0 = time.perf_counter()
    try:
        resp = await client.chat.completions.create(
            model=config['model'],
            messages=[{'role': 'user', 'content': [
                {'type': 'text', 'text': VISION_PROMPT},
                {'type': 'image_url', 'image_url': {'url': f'data:image/jpeg;base64,{img_b64}'}}
            ]}],
            max_tokens=300
        )
        elapsed = (time.perf_counter() - t0) * 1000
        content = resp.choices[0].message.content or ''
        parsed = json.loads(content)
        return parsed, elapsed, True
    except Exception as e:
        elapsed = (time.perf_counter() - t0) * 1000
        return None, elapsed, False

print('Run async benchmark cells below with actual images present')

In [ ]:
# Placeholder results for offline viewing (replace with actual benchmark run)
import numpy as np
np.random.seed(42)

records = []
for provider, lat_mean, acc in [('llava-ollama', 4200, 0.60), ('minicpm-ollama', 2800, 0.65),
                                  ('gpt4o-vision', 1100, 0.88), ('perceptionlm', 900, 0.82)]:
    for _ in range(10):
        records.append({'provider': provider, 'latency_ms': max(100, lat_mean + np.random.randn()*lat_mean*0.2),
                        'env_correct': np.random.rand() < acc, 'amenity_f1': acc + np.random.randn()*0.1})

df = pd.DataFrame(records)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Latency
sns.violinplot(data=df, x='provider', y='latency_ms', ax=axes[0], inner='box')
axes[0].set_title('VLM Response Latency')
axes[0].set_ylabel('Latency (ms)')
axes[0].tick_params(axis='x', rotation=20)

# Accuracy
acc_df = df.groupby('provider')[['env_correct', 'amenity_f1']].mean()
acc_df.plot(kind='bar', ax=axes[1])
axes[1].set_title('Classification Accuracy & Amenity F1')
axes[1].set_ylabel('Score')
axes[1].set_ylim(0, 1)
axes[1].tick_params(axis='x', rotation=20)

# Scatter
summary = df.groupby('provider').agg({'latency_ms': 'median', 'amenity_f1': 'mean'}).reset_index()
axes[2].scatter(summary['latency_ms'], summary['amenity_f1'], s=120)
for _, row in summary.iterrows():
    axes[2].annotate(row['provider'], (row['latency_ms'], row['amenity_f1']),
                     textcoords='offset points', xytext=(5, 5), fontsize=9)
axes[2].set_title('Latency vs Amenity F1')
axes[2].set_xlabel('Median Latency (ms)')
axes[2].set_ylabel('Amenity F1')

plt.tight_layout()
plt.savefig('results_04_vlm_comparison.png', dpi=150)
plt.show()